# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

Cloning into 'SQuAD-explorer'...
remote: Enumerating objects: 5563, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.bjects:  11% (2/18)
remote: Total 5563 (delta 11), reused 17 (delta 6), pack-reused 5539 (from 1)
Receiving objects: 100% (5563/5563), 52.26 MiB | 208.00 KiB/s, done.
Resolving deltas: 100% (3563/3563), done.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [15]:
# YOUR CHANGES HERE
import pandas as pd

parsed_rows = []

for document in train_data["data"]:
    document_title = document["title"]
    
    for paragraph_index, paragraph in enumerate(document["paragraphs"]):
        parsed_rows.append({
            "document_title": document_title,
            "paragraph_index": paragraph_index,
            "paragraph_context": paragraph["context"]
        })

parsed_dataframe = pd.DataFrame(parsed_rows)

parsed_dataframe.to_csv("parsed.tsv", sep="\t", index=False)

Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [18]:
# YOUR CHANGES HERE

import numpy as np

def make_paragraph_vector(paragraph_text):
    paragraph_vector = np.zeros(1024, dtype=int)
    words = str(paragraph_text).lower().split()
    
    for word in words:
        vector_index = hash(word) % 1024
        paragraph_vector[vector_index] += 1
    
    return paragraph_vector.tolist()

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [19]:
# YOUR CHANGES HERE

parsed_dataframe["paragraph_vector_json"] = parsed_dataframe["paragraph_context"].apply(
    lambda text: json.dumps(make_paragraph_vector(text))
)

output_dataframe = parsed_dataframe[[
    "document_title",
    "paragraph_index",
    "paragraph_vector_json"
]]

output_dataframe.to_csv("paragraph-vectors.tsv.gz", sep="\t", index=False)


Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [21]:
# YOUR CHANGES HERE

questions_dataframe = pd.read_csv("questions.tsv", sep="\t")

def make_text_vector(text):
    text_vector = np.zeros(1024, dtype=int)
    words = str(text).lower().split()
    
    for word in words:
        vector_index = hash(word) % 1024
        text_vector[vector_index] += 1
    
    return text_vector.tolist()

questions_dataframe["question_vector_json"] = questions_dataframe["question"].apply(
    lambda text: json.dumps(make_text_vector(text))
)

print(questions_dataframe.head())

   question_id                                           question  \
0            1  What was the goal of the abuse of region project?   
1            4  How many satellites in the Beidou-1 constellat...   
2            7  When did Beyoncé  receive ten nominations for ...   
3           10  With which goddess did Sulla, Pompey, and Juli...   
4           13  What area is considered to have a desert clima...   

                                question_vector_json  
0  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
3  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
4  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  


Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [22]:
# YOUR CHANGES HERE

output_dataframe = questions_dataframe[["question_id", "question_vector_json"]]

output_dataframe.to_csv("question-vectors.tsv", sep="\t", index=False)

Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [23]:
# YOUR CHANGES HERE


paragraph_vectors_dataframe = pd.read_csv("paragraph-vectors.tsv.gz", sep="\t")
question_vectors_dataframe = pd.read_csv("question-vectors.tsv", sep="\t")

paragraph_matrix = np.array(
    paragraph_vectors_dataframe["paragraph_vector_json"].apply(json.loads).tolist()
)

question_matrix = np.array(
    question_vectors_dataframe["question_vector_json"].apply(json.loads).tolist()
)

top_matches = []

for question_row_index, question_row in question_vectors_dataframe.iterrows():
    question_id = question_row["question_id"]
    question_vector = question_matrix[question_row_index]
    
    euclidean_distances = np.sqrt(((paragraph_matrix - question_vector) ** 2).sum(axis=1))
    top_5_paragraph_indexes = np.argsort(euclidean_distances)[:5]
    
    for question_rank, paragraph_row_index in enumerate(top_5_paragraph_indexes, start=1):
        top_matches.append({
            "question_id": question_id,
            "question_rank": question_rank,
            "document_title": paragraph_vectors_dataframe.iloc[paragraph_row_index]["document_title"],
            "paragraph_index": paragraph_vectors_dataframe.iloc[paragraph_row_index]["paragraph_index"]
        })

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [24]:
# YOUR CHANGES HERE

question_matches_dataframe = pd.DataFrame(top_matches)

question_matches_dataframe.to_csv("question-matches.tsv", sep="\t", index=False)

Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


In [26]:

questions_dataframe = pd.read_csv("questions.tsv", sep="\t")
matches_dataframe = pd.read_csv("question-matches.tsv", sep="\t")
parsed_dataframe = pd.read_csv("parsed.tsv", sep="\t")

first_5_question_ids = (
    questions_dataframe["question_id"]
    .sort_values()
    .head(5)
    .tolist()
)

review_dataframe = (
    matches_dataframe[matches_dataframe["question_id"].isin(first_5_question_ids)]
    .merge(questions_dataframe[["question_id", "question"]], on="question_id", how="left")
    .merge(
        parsed_dataframe[["document_title", "paragraph_index", "paragraph_context"]],
        on=["document_title", "paragraph_index"],
        how="left"
    )
    .sort_values(["question_id", "question_rank"])
)

for question_id in first_5_question_ids:
    question_rows = review_dataframe[review_dataframe["question_id"] == question_id]
    print("\n" + "=" * 80)
    print("question_id:", question_id)
    print("question:", question_rows.iloc[0]["question"])
    
    for _, row in question_rows.iterrows():
        print("\nrank:", row["question_rank"])
        print("document_title:", row["document_title"])
        print("paragraph_index:", row["paragraph_index"])
        print("paragraph_context:", row["paragraph_context"])


question_id: 1
question: What was the goal of the abuse of region project?

rank: 1
document_title: MP3
paragraph_index: 11
paragraph_context: ASPEC was the joint proposal of AT&T Bell Laboratories, Thomson Consumer Electronics, Fraunhofer Society and CNET. It provided the highest coding efficiency.

rank: 2
document_title: Somalis
paragraph_index: 46
paragraph_context: All of these traditions, including festivals, martial arts, dress, literature, sport and games such as Shax, have immensely contributed to the enrichment of Somali heritage.

rank: 3
document_title: Buddhism
paragraph_index: 31
paragraph_context: The Twelve Nidānas describe a causal connection between the subsequent characteristics or conditions of cyclic existence, each one giving rise to the next:

rank: 4
document_title: Bern
paragraph_index: 10
paragraph_context: A number of congresses of the socialist First and Second Internationals were held in Bern, particularly during World War I when Switzerland was neutral; s

Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.